# Structuring Agentic Memory: does the model answer from context or from prior?

Small experiment testing whether a language model answers from memory placed in
its context or from knowledge absorbed in training, and whether the way that
memory is delivered changes the balance.

**Design.** Three crossed factors over a synthetic memory:

| factor | levels |
|---|---|
| `condition` | no_memory, relevant_only, full_memory, full_plus_random, full_plus_confusable |
| `framing` | authoritative, neutral |
| `model` | llama-3.1-8b, llama-3.3-70b, qwen3.6-27b |
  

**Run order:** every cell, top to bottom. Set `GROQ_API_KEY` as a Colab secret
(key icon in the sidebar) before starting.

## 1. Setup

In [1]:
!pip install -q groq pandas

import os
from google.colab import userdata
from groq import Groq

import json, re, time, threading, unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq()

# Confirm the key works and see what is actually available.
models = sorted(m.id for m in client.models.list().data)
print(f"{len(models)} models available:\n")
for m in models:
    print(" ", m)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 851.9 kB/s eta 0:00:00
15 models available:

  allam-2-7b
  canopylabs/orpheus-arabic-saudi
  canopylabs/orpheus-v1-english
  groq/compound
  groq/compound-mini
  llama-3.1-8b-instant
  llama-3.3-70b-versatile
  meta-llama/llama-prompt-guard-2-22m
  meta-llama/llama-prompt-guard-2-86m
  openai/gpt-oss-120b
  openai/gpt-oss-20b
  openai/gpt-oss-safeguard-20b
  qwen/qwen3.6-27b
  whisper-large-v3
  whisper-large-v3-turbo


## 2. Synthetic memory and answer key

Written before any model is run; nothing here derives from model output. All
entities fictional, no real personal data.

Entry kinds: `fictional` (invented, unknowable), `override` (contradicts a
strong prior), `bridge` (fictional fact that depends on an override, so
answering requires the override to propagate one hop), `stale`/`current`
(two entries that conflict with each other).

Each query carries explicit match terms so scoring is reproducible rather than
eyeballed. `distractor_match` catches the specific failure where the model
grabs a confusable distractor instead of the real fact.

In [2]:
SPEC = {
"memory": [
  {"id":"F1","kind":"fictional","text":"Dr. Elin Voss leads the Cartography Unit at the Meridian Institute."},
  {"id":"F2","kind":"fictional","text":"The Cartography Unit is on the fourth floor of the Halden Building."},
  {"id":"F3","kind":"fictional","text":"The Halden Building requires badge access after 19:00."},
  {"id":"F4","kind":"fictional","text":"Kestrel-7 is the Institute's long-range field survey drone."},
  {"id":"F5","kind":"fictional","text":"Tomas Aldridge is the technician responsible for maintaining Kestrel-7."},
  {"id":"F6","kind":"fictional","text":"Protocol Amber requires samples to be held at the sea-level boiling point of water for exactly two minutes."},
  {"id":"O1","kind":"override","contradicts":"capital of Australia = Canberra","text":"In this setting, the capital of Australia is Sydney."},
  {"id":"O2","kind":"override","contradicts":"water boils at 100 C","text":"In this setting, water boils at 130 degrees Celsius at sea level."},
  {"id":"O3","kind":"override","contradicts":"tallest mountain = Everest","text":"In this setting, the tallest mountain on Earth is Mount Kenya."},
  {"id":"B1","kind":"bridge","depends_on":["O1"],"text":"Dr. Voss's field office is located in the capital of Australia."},
  {"id":"B2","kind":"bridge","depends_on":["O3"],"text":"The Meridian Institute's high-altitude station sits on the tallest mountain on Earth."},
  {"id":"S1a","kind":"stale","pair":"S1","text":"Kestrel-7 is stationed at the Northfield airstrip."},
  {"id":"S1b","kind":"current","pair":"S1","text":"[2031-03-14] Kestrel-7 was relocated to the Bracken depot."},
  {"id":"S2a","kind":"stale","pair":"S2","text":"[2030-06-02] The Cartography Unit's server room is in the basement."},
  {"id":"S2b","kind":"current","pair":"S2","text":"[2031-09-20] The Cartography Unit's server room was moved to the east annex."},
  {"id":"S3a","kind":"stale","pair":"S3","text":"The night shift supervisor is Ivo Brandt."},
  {"id":"S3b","kind":"current","pair":"S3","text":"The night shift supervisor is now Sanne de Vries."},
  {"id":"S4a","kind":"stale","pair":"S4","text":"Access badges are issued by the Records Annex."},
  {"id":"S4b","kind":"current","pair":"S4","text":"This entry replaces the one above: access badges are now issued by the Halden lobby desk."},
],
"queries": [
  {"id":"Q1","tier":"direct","fact_kind":"fictional","needs":["F2"],"hops":1,
   "question":"Which floor is the Cartography Unit on?",
   "mem":"fourth floor","prior":None,
   "mem_match":["fourth","4th"],"prior_match":None,"distractor_match":["second","bramwell"]},
  {"id":"Q2","tier":"direct","fact_kind":"override","needs":["O1"],"hops":1,
   "question":"In this setting, what is the capital of Australia?",
   "mem":"Sydney","prior":"Canberra",
   "mem_match":["sydney"],"prior_match":["canberra"],"distractor_match":None},
  {"id":"Q3","tier":"direct","fact_kind":"fictional","needs":["F5"],"hops":1,
   "question":"Who maintains Kestrel-7?",
   "mem":"Tomas Aldridge","prior":None,
   "mem_match":["aldridge"],"prior_match":None,"distractor_match":["aldrich"]},
  {"id":"Q4","tier":"paraphrase","fact_kind":"fictional","needs":["F1","F2"],"hops":2,
   "question":"I need to drop a package off with Dr. Voss. Which floor should I go to?",
   "mem":"fourth floor","prior":None,
   "mem_match":["fourth","4th"],"prior_match":None,"distractor_match":["second","records annex"]},
  {"id":"Q5","tier":"paraphrase","fact_kind":"override","needs":["O2"],"hops":1,
   "question":"A kettle is at sea level in this setting. At what reading on the thermometer does the water start to boil?",
   "mem":"130 C","prior":"100 C",
   "mem_match":["130"],"prior_match":["100"],"distractor_match":None},
  {"id":"Q6","tier":"compose_fictional","fact_kind":"fictional","needs":["F1","F2","F3"],"hops":3,
   "question":"Dr. Voss plans to work until 20:00 tomorrow. Will she need badge access to stay?",
   "mem":"yes","prior":None,
   "mem_match":["yes","will need","does need","needs badge"],"prior_match":None,
   "distractor_match":None,"neg_match":["no,","will not need","does not need","not need","no badge"]},
  {"id":"Q7","tier":"compose_override","fact_kind":"override","needs":["O1","B1"],"hops":2,
   "question":"Which city is Dr. Voss's field office in?",
   "mem":"Sydney","prior":"Canberra",
   "mem_match":["sydney"],"prior_match":["canberra"],"distractor_match":None},
  {"id":"Q8","tier":"compose_override","fact_kind":"override","needs":["O2","F6"],"hops":2,
   "question":"At what temperature does Protocol Amber hold its samples?",
   "mem":"130 C","prior":"100 C",
   "mem_match":["130"],"prior_match":["100"],"distractor_match":["chill","umber"]},
  {"id":"Q9","tier":"compose_override","fact_kind":"override","needs":["O3","B2"],"hops":2,
   "question":"On which mountain is the Institute's high-altitude station?",
   "mem":"Mount Kenya","prior":"Mount Everest",
   "mem_match":["kenya"],"prior_match":["everest"],"distractor_match":None},
  {"id":"Q10","tier":"unanswerable","fact_kind":"none","needs":[],"hops":0,
   "question":"Which unit does Dr. Priya Raman lead at the Institute?",
   "mem":"NOT IN MEMORY","prior":None,
   "mem_match":[],"prior_match":None,"distractor_match":None,"expect_abstain":True},
  {"id":"Q11","tier":"partially_specified","fact_kind":"fictional","needs":["F2"],"hops":1,
   "question":"What is the room number of the Cartography Unit?",
   "mem":"NOT IN MEMORY","prior":None,
   "mem_match":[],"prior_match":None,"distractor_match":None,"expect_abstain":True},
  {"id":"Q12","tier":"ambiguous","fact_kind":"fictional","needs":["F1"],"hops":1,
   "question":"Where does Voss work?",
   "mem":"Cartography Unit / fourth floor, or a disambiguation request","prior":None,
   "mem_match":["cartography","fourth","4th"],"prior_match":None,
   "distractor_match":["records annex","second floor","facilities"],"always_review":True},
],
"distractors": {
  "random":["The Institute cafeteria serves lentil soup on Wednesdays.",
            "The archive catalogue was migrated to a new system last spring.",
            "Bicycle racks are located behind the east loading dock.",
            "The visitor car park closes at 22:00 on weekends.",
            "Fern Rossi coordinates the summer lecture series.",
            "The library subscribes to eleven cartography journals."],
  "confusable":["Alexis Voss-Meyer works in the Records Annex on the second floor.",
            "Marek Voss is the facilities coordinator for the Halden Building.",
            "The Climatology Unit is on the fourth floor of the Bramwell Building.",
            "Kestrel-4 was decommissioned and is stored at Northfield.",
            "Tomas Aldrich manages procurement for the Cartography Unit.",
            "Protocol Umber requires samples to be chilled for two minutes."],
},
}

PAIRS = {
 "S1":("S1a","S1b","one dated, one undated","Bracken depot",
       "Where is Kestrel-7 currently stationed?",["bracken"],["northfield"]),
 "S2":("S2a","S2b","both dated","east annex",
       "Where is the Cartography Unit's server room?",["annex"],["basement"]),
 "S3":("S3a","S3b","no dates, 'now' only","Sanne de Vries",
       "Who is the night shift supervisor?",["sanne","vries"],["ivo","brandt"]),
 "S4":("S4a","S4b","explicit supersession","Halden lobby desk",
       "Who issues access badges?",["lobby"],["records annex"]),
}
for pair,(a,b,cue,ans,q,memm,stalem) in PAIRS.items():
    for order in ("current_last","current_first"):
        SPEC["queries"].append({
            "id":f"Q{pair}_{'CL' if order=='current_last' else 'CF'}",
            "tier":"staleness","fact_kind":"fictional","needs":[a,b],"hops":1,
            "question":q,"mem":ans,"prior":None,
            "mem_match":memm,"prior_match":None,"stale_match":stalem,
            "distractor_match":None,"cue":cue,"pair":pair,"order":order,
            "pin_order":(a,b) if order=="current_last" else (b,a)})

In [3]:
# Validation: catches errors that would invalidate rather than crash the run.
def validate(spec):
    ids = [e["id"] for e in spec["memory"]]; known = set(ids); problems = []
    for i, c in Counter(ids).items():
        if c > 1: problems.append(f"duplicate memory id {i}")
    for q in spec["queries"]:
        for n in q["needs"]:
            if n not in known: problems.append(f"{q['id']} needs unknown {n}")
    if len(spec["distractors"]["random"]) != len(spec["distractors"]["confusable"]):
        problems.append("distractor pools differ in length")
    corpus = " ".join(e["text"] for e in spec["memory"]) + " ".join(
        spec["distractors"]["random"] + spec["distractors"]["confusable"])
    if "Priya" in corpus:
        problems.append("Q10 compromised: 'Priya' appears in the memory")
    return problems

problems = validate(SPEC)
print(f"memory entries : {len(SPEC['memory'])}")
print(f"queries        : {len(SPEC['queries'])}")
for t, n in Counter(q["tier"] for q in SPEC["queries"]).most_common():
    print(f"    {t:<22} {n}")
print("\nfact kind      :", dict(Counter(q["fact_kind"] for q in SPEC["queries"])))
print("validation     :", "OK" if not problems else "FAILED")
for p in problems: print("   !!", p)

memory entries : 19
queries        : 20
    staleness              8
    direct                 3
    compose_override       3
    paraphrase             2
    compose_fictional      1
    unanswerable           1
    partially_specified    1
    ambiguous              1

fact kind      : {'fictional': 14, 'override': 5, 'none': 1}
validation     : OK


## 3. Prompt construction

The system prompt is byte-identical across conditions except for one sentence
(the framing manipulation). Distractors are interleaved rather than appended, so
real facts are not systematically early in the context.

Staleness pairs have their relative order **pinned**: which of two conflicting
entries appears first would otherwise be random, an uncontrolled variable
sitting directly on top of the effect being measured.

In [4]:
import random

CONDITIONS = ("no_memory", "relevant_only", "full_memory",
              "full_plus_random", "full_plus_confusable")

SYSTEM_TEMPLATE = (
    "You are answering questions about a specific fictional setting.\n"
    "A MEMORY block may be provided. {precedence}\n"
    "\n"
    "Rules:\n"
    "1. Answer in at most one short sentence.\n"
    "2. If the MEMORY does not determine the answer, reply exactly: NOT IN MEMORY\n"
    "3. Do not explain your reasoning."
)
FRAMINGS = {
    "authoritative": "It describes how things are in this setting, and it takes "
                     "precedence over anything you believe otherwise.",
    "neutral":       "It contains notes about this setting.",
}
FILLER_FOR_EMPTY_NEEDS = ("F4", "F5")
SEED = 20260814


def enforce_order(lines, lookup, pin):
    """Swap a pinned pair into the required relative order, leaving all other
    lines where the shuffle put them."""
    if not pin: return lines
    first, second = lookup[pin[0]], lookup[pin[1]]
    if first not in lines or second not in lines: return lines
    i, j = lines.index(first), lines.index(second)
    if i > j: lines[i], lines[j] = lines[j], lines[i]
    return lines


def build_prompts(spec, seed=SEED):
    lookup  = {e["id"]: e["text"] for e in spec["memory"]}
    all_ids = [e["id"] for e in spec["memory"]]
    pools = {"full_plus_random":     spec["distractors"]["random"],
             "full_plus_confusable": spec["distractors"]["confusable"]}
    prompts = []
    for q in spec["queries"]:
        for cond in CONDITIONS:
            if cond == "no_memory":       ids = []
            elif cond == "relevant_only": ids = list(q["needs"]) or list(FILLER_FOR_EMPTY_NEEDS)
            else:                         ids = all_ids
            distractors = pools.get(cond, [])
            lines = [lookup[i] for i in ids] + list(distractors)
            random.Random(f"{seed}:{q['id']}:{cond}").shuffle(lines)
            lines = enforce_order(lines, lookup, q.get("pin_order"))
            block = "\n".join(f"- {l}" for l in lines)
            user = (f"MEMORY:\n{block}\n\nQuestion: {q['question']}"
                    if block else f"Question: {q['question']}")
            for framing, sentence in FRAMINGS.items():
                system = SYSTEM_TEMPLATE.format(precedence=sentence)
                prompts.append({
                    "prompt_id": f"{q['id']}__{cond}__{framing}",
                    "query_id": q["id"], "condition": cond, "framing": framing,
                    "tier": q["tier"], "fact_kind": q["fact_kind"],
                    "hops": q["hops"], "question": q["question"],
                    "memory_ids": ids, "system": system, "user": user,
                    "n_entries": len(ids) + len(distractors),
                    "n_chars": len(system) + 1 + len(user),
                    "cue": q.get("cue"), "pair": q.get("pair"),
                    "order": q.get("order"),
                })
    return prompts


PROMPTS = build_prompts(SPEC)
Q_BY_ID = {q["id"]: q for q in SPEC["queries"]}

print(f"{len(PROMPTS)} prompts = {len(SPEC['queries'])} queries "
      f"x {len(CONDITIONS)} conditions x {len(FRAMINGS)} framings\n")
print(f"{'condition':<24}{'entries':>9}{'chars':>9}")
print("-" * 42)
for c in CONDITIONS:
    s = [p for p in PROMPTS if p["condition"] == c]
    print(f"{c:<24}{sum(x['n_entries'] for x in s)/len(s):>9.1f}"
          f"{sum(x['n_chars'] for x in s)/len(s):>9.0f}")

a = next(p for p in PROMPTS if p["prompt_id"] == "Q7__relevant_only__authoritative")
n = next(p for p in PROMPTS if p["prompt_id"] == "Q7__relevant_only__neutral")
print("\nuser message identical across framings:", a["user"] == n["user"])

200 prompts = 20 queries x 5 conditions x 2 framings

condition                 entries    chars
------------------------------------------
no_memory                     0.0      385
relevant_only                 1.8      510
full_memory                  19.0     1679
full_plus_random             25.0     2024
full_plus_confusable         25.0     2074

user message identical across framings: True


## 4. Budget check (no API calls)

Groq free tier is roughly 30 RPM / 6,000 TPM, with per-model daily caps applied
at organisation level. Token-per-minute usually binds before requests-per-minute,
especially for a reasoning model whose output is ~60x the others'.

The run is tiered rather than uniform: full factorial on the cheap model,
replication on a subset with the others. Limits below are from published
summaries and may be stale — the runner reads the `x-ratelimit-*` headers, which
are authoritative.

In [5]:
CHARS_PER_TOKEN = 3.8              # measured: 511 chars -> 141 ptok
OBSERVED_CTOK = {"llama-3.1-8b-instant": 15,
                 "llama-3.3-70b-versatile": 10,
                 "qwen/qwen3.6-27b": 950}
LIMITS = {"llama-3.1-8b-instant":    {"rpd": 14400, "tpd": 500_000, "tpm": 6_000},
          "llama-3.3-70b-versatile": {"rpd":  1000, "tpd": 100_000, "tpm": 12_000},
          "qwen/qwen3.6-27b":        {"rpd":  1000, "tpd": 100_000, "tpm": 6_000}}

K_MAIN = 3     # samples per cell on the main model; raise to 4 if you have time

# no_memory is dropped from the staleness tier: with no memory there is no
# conflict to resolve, so that cell was never meaningful.
PLAN = {
 "llama-3.1-8b-instant":    {"tiers":"ALL","conditions":"ALL","orders":"ALL","k":K_MAIN},
 "llama-3.3-70b-versatile": {"tiers":["staleness"],
                             "conditions":["relevant_only","full_memory","full_plus_confusable"],
                             "orders":"ALL","k":2},
 "qwen/qwen3.6-27b":        {"tiers":["staleness"],
                             "conditions":["relevant_only","full_plus_confusable"],
                             "orders":["current_last"],"k":2},
}

def select(prompts, cfg):
    out = []
    for p in prompts:
        if p["tier"] == "staleness" and p["condition"] == "no_memory": continue
        if cfg["tiers"] != "ALL" and p["tier"] not in cfg["tiers"]: continue
        if cfg["conditions"] != "ALL" and p["condition"] not in cfg["conditions"]: continue
        if cfg["orders"] != "ALL" and p.get("order") and p["order"] not in cfg["orders"]: continue
        out.append(p)
    return out

print(f"{'model':<26}{'calls':>7}{'tokens':>10}{'RPD':>7}{'TPD':>7}{'rpm':>6}{'min':>7}")
print("-" * 76)
total_calls = total_min = 0; all_ok = True
for model, cfg in PLAN.items():
    sub = select(PROMPTS, cfg); calls = len(sub) * cfg["k"]
    per_call = sum(p["n_chars"]/CHARS_PER_TOKEN for p in sub)/len(sub) + OBSERVED_CTOK[model]
    tokens = per_call * calls
    lim = LIMITS[model]
    # Pace to whichever limit binds first: 30 RPM, or TPM / tokens-per-call.
    rpm = max(1, min(28, int(0.85 * lim["tpm"] / per_call)))
    cfg["rpm"] = rpm
    mins = calls / rpm
    pr, pt = 100*calls/lim["rpd"], 100*tokens/lim["tpd"]
    ok = pr < 80 and pt < 80; all_ok &= ok
    total_calls += calls; total_min += mins
    print(f"{model:<26}{calls:>7}{tokens:>10,.0f}{pr:>6.0f}%{pt:>6.0f}%"
          f"{rpm:>6}{mins:>7.0f}  {'OK' if ok else 'OVER'}")

print("-" * 76)
print(f"{'TOTAL':<26}{total_calls:>7}{'':>10}{'':>7}{'':>7}{'':>6}{total_min:>7.0f} min")
print("\nverdict:", "within budget" if all_ok else "OVER -- lower K_MAIN or trim PLAN")

model                       calls    tokens    RPD    TPD   rpm    min
----------------------------------------------------------------------------
llama-3.1-8b-instant          552   214,228     4%    43%    13     42  OK
llama-3.3-70b-versatile        96    36,644    10%    37%    26      4  OK
qwen/qwen3.6-27b               32    41,221     3%    41%     3     11  OK
----------------------------------------------------------------------------
TOTAL                         680                                   57 min

verdict: within budget


## 5. Runner

Groq exposes no logprobs and no `n>1`, so the dependent variable is the
proportion of `k` samples that are memory-consistent. That measures sampling
stability rather than the model's token distribution — a weaker instrument, and
a limitation for the write-up.

No seed is passed: with `temperature > 0` a fixed seed would make all `k`
samples identical and collapse the DV to a single draw.

Raw responses are written to disk verbatim. Scoring is separate, so a scoring
bug never costs a re-run. The run is resumable: Colab disconnects, and without
this a drop would cost the whole run.

In [6]:
THINK_BLOCK = re.compile(r"<think>.*?</think>\s*", re.DOTALL | re.IGNORECASE)
THINK_OPEN  = re.compile(r"<think>", re.IGNORECASE)
RETRYABLE_NAMES  = {"RateLimitError","APIConnectionError","APITimeoutError","InternalServerError"}
RETRYABLE_STATUS = {408, 429, 500, 502, 503, 504}


class RateLimiter:
    """Global pacer. Free tier throttles at organisation level, so one limiter
    across all worker threads, not one per thread."""
    def __init__(self, rpm):
        self.interval = 60.0 / rpm
        self.lock = threading.Lock()
        self.next_ok = 0.0
    def wait(self):
        with self.lock:
            now = time.monotonic()
            sleep_for = max(0.0, self.next_ok - now)
            self.next_ok = max(now, self.next_ok) + self.interval
        if sleep_for: time.sleep(sleep_for)


def is_retryable(exc):
    """Identify transient failures by name or status, so we do not depend on
    which exception classes this SDK version happens to export."""
    if type(exc).__name__ in RETRYABLE_NAMES: return True
    return getattr(exc, "status_code", None) in RETRYABLE_STATUS


def parse_answer(raw_text, finish_reason):
    """Return (answer, valid). A truncated response is a MISSING answer, not a
    wrong one; scoring it as OTHER would understate the model."""
    text = THINK_BLOCK.sub("", raw_text or "").strip()
    if THINK_OPEN.search(text):            # unclosed tag = cut mid-trace
        return "", False
    if finish_reason == "length" or not text:
        return text, False
    return text, True


def completed_keys(path):
    keys = set()
    try:
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                try: row = json.loads(line)
                except json.JSONDecodeError: continue   # partial final line
                if row.get("error") is None:
                    keys.add((row["prompt_id"], row["model"], row["sample_idx"]))
    except FileNotFoundError: pass
    return keys


def call_once(model, prompt, sample_idx, limiter, temperature, max_tokens,
              max_retries=6):
    base = {k: prompt[k] for k in ("prompt_id","query_id","condition","framing",
                                   "tier","fact_kind","hops","cue","pair","order")}
    base |= {"model": model, "sample_idx": sample_idx, "temperature": temperature}
    empty = {"raw_text":"","answer":"","valid":False,"prompt_tokens":0,
             "completion_tokens":0,"finish_reason":"","latency_s":0.0}
    for attempt in range(max_retries):
        limiter.wait()
        started = time.perf_counter()
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role":"system","content":prompt["system"]},
                          {"role":"user","content":prompt["user"]}],
                temperature=temperature, max_completion_tokens=max_tokens)
            ch = resp.choices[0]
            raw = ch.message.content or ""
            finish = ch.finish_reason or ""
            answer, valid = parse_answer(raw, finish)
            return {**base, "raw_text": raw, "answer": answer, "valid": valid,
                    "prompt_tokens": resp.usage.prompt_tokens,
                    "completion_tokens": resp.usage.completion_tokens,
                    "finish_reason": finish,
                    "latency_s": round(time.perf_counter()-started, 3),
                    "error": None}
        except Exception as exc:
            if not is_retryable(exc) or attempt == max_retries - 1:
                return {**base, **empty, "error": repr(exc)[:300]}
            time.sleep(min(2 ** attempt + 0.5, 30))


def run_model(model, prompts, cfg, out_path, temperature=0.7,
              max_tokens=3072, workers=4):
    limiter = RateLimiter(cfg["rpm"])
    done = completed_keys(out_path)
    todo = [(p, i) for p in prompts for i in range(cfg["k"])
            if (p["prompt_id"], model, i) not in done]
    if not todo:
        print(f"  {model}: nothing to do"); return
    print(f"  {model}: {len(todo)} calls at {cfg['rpm']} rpm "
          f"(~{len(todo)/cfg['rpm']:.0f} min)")
    lock, n_done, n_err, n_bad = threading.Lock(), 0, 0, 0
    t0 = time.perf_counter()
    with open(out_path, "a", encoding="utf-8") as fh, \
         ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(call_once, model, p, i, limiter,
                               temperature, max_tokens) for p, i in todo]
        for fut in as_completed(futures):
            row = fut.result()
            with lock:
                fh.write(json.dumps(row, ensure_ascii=False) + "\n"); fh.flush()
                n_done += 1
                n_err += row["error"] is not None
                n_bad += row["error"] is None and not row["valid"]
                if n_done % 50 == 0 or n_done == len(todo):
                    el = (time.perf_counter()-t0)/60
                    print(f"    {n_done}/{len(todo)}  err={n_err} invalid={n_bad}"
                          f"  {el:.1f} min", flush=True)
    print(f"    finished: {n_done} calls, {n_err} errors, {n_bad} invalid")


print("runner ready")

runner ready


## 6. Run



In [7]:
OUT = "raw_outputs.jsonl"
t0 = time.perf_counter()
for model, cfg in PLAN.items():
    run_model(model, select(PROMPTS, cfg), cfg, OUT)
print(f"\nall models done in {(time.perf_counter()-t0)/60:.1f} min -> {OUT}")

rows = [json.loads(l) for l in open(OUT)]
print(f"{len(rows)} rows | {sum(r['error'] is not None for r in rows)} errors "
      f"| {sum(not r['valid'] for r in rows)} invalid")

  llama-3.1-8b-instant: 552 calls at 13 rpm (~42 min)
    50/552  err=0 invalid=0  3.8 min
    100/552  err=0 invalid=0  7.7 min
    150/552  err=0 invalid=0  12.1 min
    200/552  err=0 invalid=0  16.0 min
    250/552  err=0 invalid=0  19.8 min
    300/552  err=0 invalid=0  24.4 min
    350/552  err=0 invalid=0  28.1 min
    400/552  err=0 invalid=0  32.0 min
    450/552  err=0 invalid=0  35.8 min
    500/552  err=0 invalid=0  39.7 min
    550/552  err=0 invalid=0  43.5 min
    552/552  err=0 invalid=0  43.7 min
    finished: 552 calls, 0 errors, 0 invalid
  llama-3.3-70b-versatile: 96 calls at 26 rpm (~4 min)
    50/96  err=0 invalid=0  1.9 min
    96/96  err=0 invalid=0  3.8 min
    finished: 96 calls, 0 errors, 0 invalid
  qwen/qwen3.6-27b: 32 calls at 3 rpm (~11 min)
    32/32  err=0 invalid=0  10.4 min
    finished: 32 calls, 0 errors, 0 invalid

all models done in 57.8 min -> raw_outputs.jsonl
680 rows | 0 errors | 0 invalid


## 7. Scoring

Outcome codes:

| code | meaning |
|---|---|
| `MEM` | answer follows the memory |
| `PRIOR` | answer follows pre-training knowledge (override items only) |
| `STALE` | answer follows the superseded memory entry (staleness items only) |
| `DISTRACTOR` | answer captured by a confusable distractor |
| `ABSTAIN` | said NOT IN MEMORY |
| `OTHER` | none of the above |
| `INVALID` | truncated or empty — excluded from rates, reported separately |

Anything the rules cannot classify cleanly is flagged `needs_review` rather than
silently guessed. Those rows get read by hand in section 9.

In [8]:
ABSTAIN_RE = re.compile(r"not\s+in\s+memory", re.IGNORECASE)

def norm(text):
    t = unicodedata.normalize("NFKD", text or "").lower()
    return re.sub(r"[^a-z0-9\s.,'-]", " ", t)

def score_one(answer, valid, q):
    if not valid: return "INVALID", False
    a = norm(answer)
    if not a.strip(): return "INVALID", False
    abstained = bool(ABSTAIN_RE.search(a))
    hit = lambda terms: bool(terms) and any(t in a for t in terms)

    if q.get("neg_match") and hit(q["neg_match"]):
        return "OTHER", True
    matched = []
    if hit(q.get("mem_match")):        matched.append("MEM")
    if hit(q.get("prior_match")):      matched.append("PRIOR")
    if hit(q.get("stale_match")):      matched.append("STALE")
    if hit(q.get("distractor_match")): matched.append("DISTRACTOR")

    if abstained and not matched: return "ABSTAIN", bool(q.get("always_review"))
    if abstained and matched:     return "ABSTAIN", True
    if len(matched) > 1:          return matched[0], True     # ambiguous
    if len(matched) == 1:         return matched[0], bool(q.get("always_review"))
    return "OTHER", True

def is_correct(code, q):
    """Correct = abstained where abstention was right, else followed memory."""
    return code == "ABSTAIN" if q.get("expect_abstain") else code == "MEM"

df = pd.DataFrame([json.loads(l) for l in open(OUT)])
scored = df.apply(lambda r: score_one(r["answer"], r["valid"],
                                      Q_BY_ID[r["query_id"]]),
                  axis=1, result_type="expand")
df["code"], df["needs_review"] = scored[0], scored[1]
df["correct"] = df.apply(lambda r: is_correct(r["code"], Q_BY_ID[r["query_id"]]),
                         axis=1)

print("outcome codes:")
for c, n in df["code"].value_counts().items():
    print(f"    {c:<12}{n:>5}  ({100*n/len(df):>5.1f}%)")
print(f"\nflagged for manual review: {df['needs_review'].sum()} "
      f"({100*df['needs_review'].mean():.1f}%)")
print(f"invalid (excluded from rates): {(~df['valid']).sum()}")

outcome codes:
    MEM           344  ( 50.6%)
    ABSTAIN       284  ( 41.8%)
    STALE          41  (  6.0%)
    OTHER          11  (  1.6%)

flagged for manual review: 33 (4.9%)
invalid (excluded from rates): 0


## 8. Results

In [9]:
ok = df[df["valid"]].copy()
main = ok[ok["model"] == "llama-3.1-8b-instant"]
non_stale = main[main["tier"] != "staleness"]

def pct(s): return f"{100*s.mean():.0f}%"

print("=" * 68)
print("Q1  Does delivery CONDITION change memory use?   [llama-3.1-8b]")
print("=" * 68)
t = non_stale.groupby("condition").agg(
    correct=("correct","mean"), ptok=("prompt_tokens","mean"), n=("correct","size"))
t = t.reindex([c for c in CONDITIONS if c in t.index])
print(f"{'condition':<24}{'memory-consistent':>19}{'mean ptok':>11}{'n':>6}")
for c, r in t.iterrows():
    print(f"{c:<24}{100*r['correct']:>18.0f}%{r['ptok']:>11.0f}{int(r['n']):>6}")

print("\n" + "=" * 68)
print("Q2  Does FRAMING change it, at near-constant size?")
print("=" * 68)
t = non_stale.groupby("framing").agg(
    correct=("correct","mean"), ptok=("prompt_tokens","mean"), n=("correct","size"))
for c, r in t.iterrows():
    print(f"{c:<24}{100*r['correct']:>18.0f}%{r['ptok']:>11.0f}{int(r['n']):>6}")

print("\n" + "=" * 68)
print("Q3  Fictional facts vs facts that CONTRADICT A PRIOR")
print("=" * 68)
t = non_stale[non_stale["fact_kind"] != "none"].groupby("fact_kind").agg(
    correct=("correct","mean"), n=("correct","size"))
for c, r in t.iterrows():
    print(f"{c:<24}{100*r['correct']:>18.0f}%{int(r['n']):>17}")
print("\nPRIOR responses (prior beat the memory):",
      int((non_stale["code"] == "PRIOR").sum()))
print("DISTRACTOR responses (confusable padding captured it):",
      int((non_stale["code"] == "DISTRACTOR").sum()))

print("\n" + "=" * 68)
print("Q4  By tier -- where is it actually hard?")
print("=" * 68)
t = main.groupby("tier").agg(correct=("correct","mean"), n=("correct","size"))
for c, r in t.sort_values("correct").iterrows():
    print(f"{c:<24}{100*r['correct']:>18.0f}%{int(r['n']):>17}")

print("\n" + "=" * 68)
print("Q5  STALENESS: which memory structure makes supersession resolvable?")
print("=" * 68)
st = ok[ok["tier"] == "staleness"]
if len(st):
    g = st.pivot_table(index="cue", columns="order", values="correct", aggfunc="mean")
    print("\ncorrect (chose the CURRENT entry), by cue x order:\n")
    print((100*g).round(0).astype("Int64").to_string())
    print("\nby model:")
    for m, r in st.groupby("model").agg(correct=("correct","mean"),
                                        n=("correct","size")).iterrows():
        print(f"    {m:<26}{100*r['correct']:>6.0f}%   n={int(r['n'])}")
    print("\nby framing:")
    for f, r in st.groupby("framing").agg(correct=("correct","mean"),
                                          n=("correct","size")).iterrows():
        print(f"    {f:<26}{100*r['correct']:>6.0f}%   n={int(r['n'])}")
    print(f"\nSTALE responses (followed the superseded entry): "
          f"{int((st['code']=='STALE').sum())} / {len(st)}")

print("\n" + "=" * 68)
print("Q6  Prompt size by condition (all models)")
print("=" * 68)
t = ok.groupby("condition").agg(ptok=("prompt_tokens","mean"),
                                ctok=("completion_tokens","mean"))
for c in [c for c in CONDITIONS if c in t.index]:
    print(f"{c:<24}  prompt {t.loc[c,'ptok']:>6.0f} tok   "
          f"completion {t.loc[c,'ctok']:>7.0f} tok")

Q1  Does delivery CONDITION change memory use?   [llama-3.1-8b]
condition                 memory-consistent  mean ptok     n
no_memory                               17%        118    72
relevant_only                           92%        146    72
full_memory                             81%        436    72
full_plus_random                        74%        507    72
full_plus_confusable                    61%        526    72

Q2  Does FRAMING change it, at near-constant size?
authoritative                           60%        353   180
neutral                                 69%        341   180

Q3  Fictional facts vs facts that CONTRADICT A PRIOR
fictional                               58%              180
override                                66%              150

PRIOR responses (prior beat the memory): 0
DISTRACTOR responses (confusable padding captured it): 0

Q4  By tier -- where is it actually hard?
compose_fictional                       17%               30
staleness      

In [12]:
# =============================================================================
# CORRECTED ANALYSIS
#   1. tier comparison excludes no_memory (it was dragging every tier down)
#   2. model comparison restricted to matched cells only
#   3. abstention rate by framing, to test why neutral > authoritative
#   4. bootstrap CIs clustered by query -- responses within a query are not
#      independent, so naive binomial CIs would be far too narrow
# =============================================================================
import numpy as np

rng = np.random.default_rng(0)

def boot_ci(frame, col="correct", n=2000):
    """Cluster bootstrap over queries: resample whole queries, not rows."""
    if frame.empty: return (np.nan, np.nan)
    groups = [g[col].values for _, g in frame.groupby("query_id")]
    means = [np.concatenate(rng.choice(groups, len(groups), replace=True)).mean()
             for _ in range(n)]
    return np.percentile(means, [2.5, 97.5]) * 100

answered = main[main["condition"] != "no_memory"]

print("=" * 72)
print("A. By tier, EXCLUDING no_memory   [llama-3.1-8b]")
print("=" * 72)
print(f"{'tier':<24}{'correct':>9}{'95% CI':>18}{'queries':>9}{'n':>6}")
for tier, g in sorted(answered.groupby("tier"), key=lambda x: x[1]["correct"].mean()):
    lo, hi = boot_ci(g)
    print(f"{tier:<24}{100*g['correct'].mean():>8.0f}%"
          f"{f'[{lo:.0f}, {hi:.0f}]':>18}{g['query_id'].nunique():>9}{len(g):>6}")

print("\n" + "=" * 72)
print("B. Why did neutral beat authoritative? -- outcome mix by framing")
print("=" * 72)
mix = (answered.groupby(["framing", "code"]).size()
       .unstack(fill_value=0).apply(lambda r: 100*r/r.sum(), axis=1))
print(mix.round(1).to_string())
for f, g in answered.groupby("framing"):
    lo, hi = boot_ci(g)
    print(f"\n  {f:<16}correct {100*g['correct'].mean():>5.0f}%  "
          f"95% CI [{lo:.0f}, {hi:.0f}]   abstain "
          f"{100*(g['code']=='ABSTAIN').mean():>5.0f}%")

print("\n" + "=" * 72)
print("C. Condition effect with CIs  (dilution vs interference)")
print("=" * 72)
for cond in CONDITIONS:
    g = main[main["condition"] == cond]
    if g.empty: continue
    lo, hi = boot_ci(g)
    print(f"{cond:<24}{100*g['correct'].mean():>8.0f}%"
          f"{f'[{lo:.0f}, {hi:.0f}]':>18}  ptok {g['prompt_tokens'].mean():>5.0f}")
r = main[main["condition"]=="full_plus_random"]["correct"].mean()
c = main[main["condition"]=="full_plus_confusable"]["correct"].mean()
print(f"\n  random -> confusable: {100*(r-c):+.0f} points at ~equal size "
      f"= INTERFERENCE, not dilution")

print("\n" + "=" * 72)
print("D. Staleness, MATCHED cells only (all 3 models saw these)")
print("=" * 72)
matched = ok[(ok["tier"]=="staleness") &
             (ok["order"]=="current_last") &
             (ok["condition"].isin(["relevant_only","full_plus_confusable"]))]
print(f"{'model':<26}{'correct':>9}{'95% CI':>18}{'n':>6}")
for m, g in matched.groupby("model"):
    lo, hi = boot_ci(g)
    print(f"{m:<26}{100*g['correct'].mean():>8.0f}%"
          f"{f'[{lo:.0f}, {hi:.0f}]':>18}{len(g):>6}")
print("\n(Unmatched comparison is invalid: qwen ran only current_last,")
print(" the easier order, and only 2 of 5 conditions.)")

print("\n" + "=" * 72)
print("E. Staleness: order vs cue  [llama-3.1-8b only, fully crossed]")
print("=" * 72)
st8 = ok[(ok["tier"]=="staleness") & (ok["model"]=="llama-3.1-8b-instant")]
for o, g in st8.groupby("order"):
    lo, hi = boot_ci(g)
    print(f"  order={o:<16}{100*g['correct'].mean():>6.0f}%  [{lo:.0f}, {hi:.0f}]")
for c, g in st8.groupby("cue"):
    lo, hi = boot_ci(g)
    print(f"  cue={c:<26}{100*g['correct'].mean():>6.0f}%  [{lo:.0f}, {hi:.0f}]")

print("\n" + "=" * 72)
print("F. Per-query, so single-query tiers are visible as such")
print("=" * 72)
pq = (answered.groupby(["tier","query_id"])["correct"].agg(["mean","size"])
      .sort_values("mean"))
for (tier, qid), r in pq.iterrows():
    print(f"  {tier:<22}{qid:<10}{100*r['mean']:>6.0f}%   n={int(r['size'])}")

A. By tier, EXCLUDING no_memory   [llama-3.1-8b]
tier                      correct            95% CI  queries     n
compose_fictional             21%          [21, 21]        1    24
staleness                     35%          [21, 50]        8   192
ambiguous                     50%          [50, 50]        1    24
compose_override              76%          [58, 92]        3    72
paraphrase                    77%          [58, 96]        2    48
direct                        92%          [88, 96]        3    72
partially_specified           96%          [96, 96]        1    24
unanswerable                  96%          [96, 96]        1    24

B. Why did neutral beat authoritative? -- outcome mix by framing
code           ABSTAIN   MEM  OTHER  STALE
framing                                   
authoritative     47.1  45.8    2.5    4.6
neutral           39.2  55.0    1.7    4.2

  authoritative   correct    55%  95% CI [41, 69]   abstain    47%

  neutral         correct    65%  95% CI 

## 9. Manual review

The task asks for model outputs to be inspected, not just aggregated. Every row
the rules could not classify cleanly is printed here. Read them and correct
`df.loc[idx, "code"]` by hand where the automatic label is wrong, then re-run
section 8.

In [10]:
review = df[df["needs_review"]]
print(f"{len(review)} rows flagged\n")

for qid, grp in review.groupby("query_id"):
    q = Q_BY_ID[qid]
    print("=" * 74)
    print(f"{qid}  [{q['tier']}]   expected -> {q['mem']}")
    print("=" * 74)
    for idx, r in grp.iterrows():
        print(f"  [{idx}] {r['model'][:22]:<24}{r['condition']:<22}"
              f"{r['framing']:<15}{r['code']}")
        print(f"       {r['answer'][:150]!r}")
    print()

# Example correction, then re-run section 8:
#   df.loc[123, "code"] = "MEM"
#   df.loc[123, "correct"] = True

33 rows flagged

Q10  [unanswerable]   expected -> NOT IN MEMORY
  [274] llama-3.1-8b-instant    relevant_only         authoritative  OTHER
       'Dr. Priya Raman leads the Artificial Intelligence Development team.'

Q11  [partially_specified]   expected -> NOT IN MEMORY
  [324] llama-3.1-8b-instant    full_plus_confusable  authoritative  OTHER
       'The Cartography Unit is on the fourth floor of the Halden Building, but its server room is in the basement, then moved to the east annex on [2031-09-2'

Q12  [ambiguous]   expected -> Cartography Unit / fourth floor, or a disambiguation request
  [330] llama-3.1-8b-instant    no_memory             authoritative  ABSTAIN
       'NOT IN MEMORY'
  [331] llama-3.1-8b-instant    no_memory             authoritative  ABSTAIN
       'NOT IN MEMORY.'
  [332] llama-3.1-8b-instant    no_memory             authoritative  ABSTAIN
       'NOT IN MEMORY'
  [333] llama-3.1-8b-instant    no_memory             neutral        ABSTAIN
       'NOT IN MEMORY

In [15]:
# =============================================================================
# FIXES + Q12 RESOLUTION
#   1. condition effect on non-staleness items (the correct slice)
#   2. tense confound in the staleness pairs
#   3. Q12: "The Meridian Institute" is TRUE but not at the granularity asked.
#      Scored UNDERSPECIFIED, not MEM and not OTHER -- the distinction is the
#      point of the item.
# =============================================================================
UNDERSPEC = [336, 338, 340, 341, 343, 350, 351, 352]
df.loc[df.index.isin(UNDERSPEC), "code"] = "UNDERSPECIFIED"
df.loc[df.index.isin(UNDERSPEC), "correct"] = False
df.loc[df["needs_review"] & (df["query_id"] == "Q12"), "needs_review"] = False

ok = df[df["valid"]].copy()
main = ok[ok["model"] == "llama-3.1-8b-instant"]
ns = main[(main["tier"] != "staleness") & (main["condition"] != "no_memory")]

print("=" * 70)
print("A. CONDITION on non-staleness items  [the interference test]")
print("=" * 70)
for c in CONDITIONS:
    g = main[(main["tier"] != "staleness") & (main["condition"] == c)]
    if g.empty: continue
    lo, hi = boot_ci(g)
    print(f"{c:<24}{100*g['correct'].mean():>7.0f}%  [{lo:>3.0f},{hi:>3.0f}]"
          f"   ptok {g['prompt_tokens'].mean():>4.0f}   n={len(g)}")
r = main[(main["tier"]!="staleness") & (main["condition"]=="full_plus_random")]["correct"].mean()
c = main[(main["tier"]!="staleness") & (main["condition"]=="full_plus_confusable")]["correct"].mean()
print(f"\n  random -> confusable: {100*(r-c):+.0f} pts at +21 tokens (+4%)")

print("\n" + "=" * 70)
print("B. TENSE CONFOUND in the staleness pairs  [llama-3.1-8b]")
print("=" * 70)
PAST = {"S1", "S2"}   # current entry written as "was relocated" / "was moved"
st8 = ok[(ok["tier"] == "staleness") & (ok["model"] == "llama-3.1-8b-instant")].copy()
st8["tense"] = st8["pair"].map(lambda p: "current=PAST" if p in PAST else "current=PRESENT")
st8["dated"] = st8["pair"].map(lambda p: "dates" if p in {"S1","S2"} else "no dates")
for t, g in st8.groupby("tense"):
    lo, hi = boot_ci(g)
    print(f"  {t:<20}{100*g['correct'].mean():>7.0f}%  [{lo:.0f},{hi:.0f}]  n={len(g)}")
print("\n  NOTE: tense and dates are perfectly collinear here (S1,S2 have both).")
print("  The design cannot separate them -- that is the follow-up experiment.")

print("\n" + "=" * 70)
print("C. Outcome mix, all non-staleness llama-8b answered cells")
print("=" * 70)
print((100*ns["code"].value_counts(normalize=True)).round(1).to_string())

print("\n" + "=" * 70)
print("D. Confabulation rate where the memory could not support an answer")
print("=" * 70)
for qid in ("Q10", "Q11"):
    g = main[(main["query_id"] == qid) & (main["condition"] != "no_memory")]
    print(f"  {qid}: abstained {100*(g['code']=='ABSTAIN').mean():>5.0f}%   "
          f"confabulated {100*(g['code']=='OTHER').mean():>5.0f}%   n={len(g)}")
print("\n  Rule 2 explicitly cues 'NOT IN MEMORY', so these are near-ceiling")
print("  by construction. Compliance, not judgement.")

A. CONDITION on non-staleness items  [the interference test]
no_memory                    17%  [  0, 42]   ptok  118   n=72
relevant_only                92%  [ 81,100]   ptok  146   n=72
full_memory                  81%  [ 61, 96]   ptok  436   n=72
full_plus_random             74%  [ 56, 89]   ptok  507   n=72
full_plus_confusable         61%  [ 39, 85]   ptok  526   n=72

  random -> confusable: +12 pts at +21 tokens (+4%)

B. TENSE CONFOUND in the staleness pairs  [llama-3.1-8b]
  current=PAST             20%  [9,29]  n=96
  current=PRESENT          50%  [32,70]  n=96

  NOTE: tense and dates are perfectly collinear here (S1,S2 have both).
  The design cannot separate them -- that is the follow-up experiment.

C. Outcome mix, all non-staleness llama-8b answered cells
code
MEM               60.8
ABSTAIN           35.8
UNDERSPECIFIED     2.8
OTHER              0.7

D. Confabulation rate where the memory could not support an answer
  Q10: abstained    96%   confabulated     4%   n=24
 

## 10. Export deliverables

In [16]:
with open("answer_key.json", "w", encoding="utf-8") as fh:
    json.dump(SPEC, fh, indent=2, ensure_ascii=False)
with open("prompts.jsonl", "w", encoding="utf-8") as fh:
    for p in PROMPTS:
        fh.write(json.dumps(p, ensure_ascii=False) + "\n")
df.to_csv("scored.csv", index=False)

summary = (df[df["valid"]].groupby(["model","condition","framing","tier"])
           .agg(memory_consistent=("correct","mean"),
                mean_prompt_tokens=("prompt_tokens","mean"),
                n=("correct","size")).reset_index())
summary.to_csv("summary.csv", index=False)

for f in ["answer_key.json","prompts.jsonl","raw_outputs.jsonl",
          "scored.csv","summary.csv"]:
    print(f"  {f:<24}{os.path.getsize(f)/1024:>8.1f} KB")

# from google.colab import files
# for f in ["answer_key.json","prompts.jsonl","raw_outputs.jsonl",
#           "scored.csv","summary.csv"]:
#     files.download(f)

  answer_key.json             13.6 KB
  prompts.jsonl              352.7 KB
  raw_outputs.jsonl          452.2 KB
  scored.csv                 261.3 KB
  summary.csv                  6.9 KB


---

## Appendix: pilot probes that shaped this design

Recorded because the decisions below are otherwise unmotivated, and the
write-up needs them.

**Probe 1 — Q7, `relevant_only` vs `full_plus_confusable`, 12 calls.** 11/12
answered Sydney. No flip. Override items were at ceiling even with three
competing Voss entities in context.

**Diagnosis.** The original system prompt said the memory *"takes precedence
over anything you believe otherwise"* — itself a strong context-faithfulness
instruction, handing the models the answer to the conflict. That sentence was
promoted from a constant to a factor (`framing`).

**Probe 2 — Q6/Q10/Q11/Q13, hardest condition, 48 calls.**

| query | result |
|---|---|
| Q6 (3-hop fictional) | llama-8b failed 2/4 (`NOT IN MEMORY`); 70b and Qwen 4/4. Chain dropped under interference — not the prior winning |
| Q10 (unanswerable) | 12/12 abstained. Ceiling, but Rule 2 explicitly cues `NOT IN MEMORY`, so this measures compliance more than judgement |
| Q11 (partial) | 11/12 abstained; one deflection (gave the floor instead of the room) |
| Q13 (staleness) | **6/11 current, 4/11 stale.** Near coin flip. authoritative 5/6 vs neutral 2/5 |

**Consequence.** Staleness was the only tier off ceiling, so it grew from one
query into a 4x2 sub-design crossing the resolution *cue* against the *order*
the conflicting entries appear in. Entry order had been left to the shuffle —
an uncontrolled variable sitting directly on top of the effect being measured —
and is now pinned.

**Also fixed.** Qwen hit `finish_reason="length"` with an unclosed `<think>`
tag, so the strip regex returned raw reasoning as the answer. Scored naively
that becomes a wrong answer when the model never produced one. Budget raised to
3072; such rows are flagged `valid=False` and excluded from rates.